In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp /content/drive/MyDrive/image/images.zip -d /content/images.zip
!unzip -q images.zip

In [ ]:
!rm /content/images.zip

In [ ]:
import os
A=os.listdir('/content')
A

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
s='/content/images/*'
path=glob.glob(s)
import re
def sorted_alphanumeric(data):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [ convert(c) for c in re.split('([0-9]+)', key) ]
    return sorted(data, key=alphanum_key)

path1=sorted_alphanumeric(path)
path1 = [item.replace('p','_') for item in path1]
data=pd.read_excel('/content/drive/MyDrive/image/data_c.xlsx')
data
len(path1)
print(path1)
type(path1)
path1=np.array(path1)
path1.shape
path1[0]

In [ ]:
path1=np.array(path1)
data=np.array(data)
data
len(data)

In [ ]:
df=pd.DataFrame({'imgpath':path1,'Porosity':data[:,0],'throat radius':data[:,1],'pore radius':data[:,2],'pore_connection_number':data[:,3],'pore shape factor':data[:,4]})

In [ ]:
df

In [ ]:
df_new=df[df['Porosity'] <0.35 ]
df_new.shape
df_new

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np
data2=np.array(df_new)
scaler = StandardScaler()
data1= scaler.fit_transform(data2[:,1:])
print(data1)
print(type(data1))
data1.shape

In [ ]:
data2

In [ ]:
df=pd.DataFrame({'imgpath':df_new['imgpath'],'Porosity':data1[:,0],'throat radius':data1[:,1],'pore radius':data1[:,2],'pore_connection_number':data1[:,3],'pore shape factor':data1[:,4]})
df

In [ ]:
data_t_general=df.sample(frac=0.9,random_state=1)
df_valid=data_t_general.sample(frac=0.15,random_state=2)
df_train=data_t_general.loc[~data_t_general.index.isin(df_valid.index)]
df_test =df.loc[~df.index.isin(data_t_general.index)]
len(df_train)

In [ ]:
def Data_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img = []
                y_batch=[]
                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    Porosity=np.array(df.Porosity)[idd]
                    throat_radius=np.array(df['throat radius'])[idd]
                    pore_radius=np.array(df['pore radius'])[idd]
                    pore_connection_number=np.array(df.pore_connection_number)[idd]
                    pore_shape_factor=np.array(df['pore shape factor'])[idd]
                    y_1=np.concatenate((Porosity, throat_radius,pore_radius,pore_connection_number, pore_shape_factor),axis=None)
                    y=np.array([y_1])
                    x_batch_img.append(img_1)
                    y_batch.append(y)


                x_batch_img = np.array(x_batch_img)
                y_batch= np.array(y_batch)
                y_batch=y_batch.reshape(-1,5)
              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img , y_batch

In [ ]:
from PIL import Image as im
img = open(np.array(df.imgpath)[1],'rb').read()
img_1=np.frombuffer(img,dtype=np.uint8)
img_1=img_1.reshape(100,100,100)
i=0
for img in img_1:
    data = im.fromarray((img*255).astype(np.uint8),mode='L')
    data.save('img_rock.jpeg')
    if i==0:
      break
    i+=1
print(img)

In [ ]:
import cv2
import numpy as np

image = cv2.imread('/content/img_rock.jpeg')

def apply_warping(image):
    height, width = image.shape[:2]

    # Define a random warping transformation
    src_points = np.float32([[0, 0], [width - 1, 0], [0, height - 1], [width - 1, height - 1]])
    dst_points = np.float32([[0, 0], [width - 1, 100], [50, height - 1], [width - 50, height - 1]])
    warp_matrix = cv2.getPerspectiveTransform(src_points, dst_points)

    # Apply the warping transformation
    warped_image = cv2.warpPerspective(image, warp_matrix, (width, height))

    return warped_image

# Example usage
#original_image = cv2.imread('path/to/your/binary_image.jpg', cv2.IMREAD_GRAYSCALE)
warped_image = apply_warping(image)

data2 = im.fromarray((warped_image))
data2.save('img_rock_warped.jpeg')

In [ ]:
pip install elasticdeform

In [ ]:
import numpy as np
import elasticdeform
import cv2

def apply_elastic_distortions(image):
    # Define elastic distortions parameters
    alpha = 50  # Deformation strength
    sigma = 5   # Smoothing parameter

    # Apply elastic distortions

    distorted_image = elasticdeform.deform_random_grid(np.array(image), sigma=25, points=3, order=4, mode='constant', cval=0.0, crop=None, prefilter=False, axis=None, affine=None, rotate=None, zoom=None)
    return distorted_image

# Example usage
#original_image = cv2.imread('/content/img_rock.jpeg', cv2.IMREAD_GRAYSCALE)
distorted_image = apply_elastic_distortions(img)
data3 = im.fromarray((distorted_image*255).astype(np.uint8))
data3.save('img_rock_distored.jpeg')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import elasticdeform

def apply_displacement(image, displacement, order=3, mode='constant', cval=0.0, crop=None):
    # Convert the image to a NumPy array
    img_array = np.array(image)

    # Apply displacement to the image using elasticdeform.deform_grid
    distorted_image = elasticdeform.deform_grid(
        img_array,
        displacement,
        order=order,
        mode=mode,
        cval=cval,
        crop=crop
    )

    return distorted_image

# Example usage
# Create a simple binary image (replace with your binary image)
binary_image = np.zeros((100, 100))
binary_image[30:70, 40:60] = 1

# Generate a random displacement field (replace with your displacement field)
displacement_field = np.random.rand(*binary_image.shape) * 10 - 5

# Apply displacement to the binary image
distorted_image = apply_displacement(binary_image, displacement_field)

# Visualize the original and distorted images
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(binary_image, cmap='gray')
plt.title('Original Image')

plt.subplot(1, 2, 2)
plt.imshow(distorted_image, cmap='gray')
plt.title('Distorted Image with Displacement')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a blank image (assume grayscale for simplicity)
image_size = 100
image = np.zeros((image_size, image_size))

# Generate a simple displacement field (movement to the right)
displacement_field = np.zeros((image_size, image_size, 2))
displacement_field[:, :, 0] = 10  # Move all pixels 10 units to the right

# Apply displacement to the image
displaced_image = np.zeros_like(image)
for i in range(image_size):
    for j in range(image_size):
        displacement = displacement_field[i, j]
        new_i, new_j = i + int(displacement[0]), j + int(displacement[1])
        if 0 <= new_i < image_size and 0 <= new_j < image_size:
            displaced_image[new_i, new_j] = image[i, j]

# Visualize the original and displaced images
plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title('Original Image')

plt.subplot(1, 2, 2)
plt.imshow(displaced_image)
plt.title('Displaced Image')

plt.show()


In [ ]:
import numpy, imageio, elasticdeform
X = numpy.zeros((200, 300))
X[::10, ::10] = 1

# apply deformation with a random 3 x 3 grid
X_deformed = elasticdeform.deform_random_grid(X, sigma=25, points=3)

imageio.imsave('test_X.png', X)
imageio.imsave('test_X_deformed.png', X_deformed)

In [ ]:
import numpy as np
import elasticdeform
import matplotlib.pyplot as plt

# Create a simple 2D image (grayscale)
image_size = 100
image = np.zeros((image_size, image_size))
image[30:70, 40:60] = 1 # Add a rectangle for visualization

# Generate a displacement field using deform_grid
def generate_displacement_field(shape, scale=20):
    displacement_field = elasticdeform.deform_grid(np.zeros(shape))
    return displacement_field * scale

displacement_field = generate_displacement_field(image.shape)

# Apply elastic deformation using deform_grid
deformed_image = elasticdeform.deform_grid(image, displacement_field)

# Visualize the original and deformed images
plt.figure(figsize=(10, 4))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap='gray')
plt.title('Original Image')

plt.subplot(1, 3, 2)
plt.imshow(displacement_field[0], cmap='gray')
plt.title('Displacement Field')

plt.subplot(1, 3, 3)
plt.imshow(deformed_image, cmap='gray')
plt.title('Deformed Image')

plt.show()

In [ ]:
####################################elastic###################################################

In [ ]:
def Data_predict_generator(df, batch_size, augment=False):

    while True:

            for start in range(0, df.shape[0], batch_size):
                x_batch_img_2 = []

                end = min(start + batch_size, df.shape[0])

                for idd in range(start,end):
                    img = open(np.array(df.imgpath)[idd],'rb').read()
                    img_1=np.frombuffer(img,dtype=np.uint8)
                    img_1=img_1.reshape(100,100,100,1)
                    x_batch_img_2.append(img_1)



                x_batch_img_2 = np.array(x_batch_img_2)

              #y_batch = to_categorical(y_batch,5)
                yield x_batch_img_2

In [ ]:
batch_size=10
train_gen= Data_generator(df_train,batch_size)
valid_gen= Data_generator(df_valid,batch_size)
train_gen_pre=Data_predict_generator(df_train,batch_size)
valid_gen_pre=Data_predict_generator(df_valid,batch_size)
test_gen= Data_generator(df_test,batch_size)
test_gen_pre=Data_predict_generator(df_test,batch_size)

In [ ]:
import math
ntrain, nvalid, ntest = df_train.shape[0], df_valid.shape[0],  df_test.shape[0]
nbatches_train=math.ceil(df_train.shape[0]/batch_size)
nbatches_valid=math.ceil(df_valid.shape[0]/batch_size)
nbatches_test=math.ceil( df_test.shape[0]/batch_size)
nworkers=1

In [ ]:
print(nbatches_valid,nbatches_train,nbatches_test)

In [ ]:

from keras.models import Sequential
from keras.layers import Conv3D,MaxPooling3D,Dropout,Flatten,Dense,BatchNormalization,ReLU
from tensorflow.keras import initializers
from keras.callbacks import *
import tensorflow as tf
from keras.optimizers import *
from keras.optimizers import Adam

In [ ]:
model=Sequential()
model.add(Conv3D(16,kernel_size=(7,7,7),input_shape=(100,100,100,1),padding="same",activation='relu',kernel_initializer=initializers.RandomNormal(stddev=0.01),))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(32,kernel_size=(5,5,5),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(64,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(128,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
#model.add(Dropout(0.2))
model.add(MaxPooling3D(pool_size=(2,2,2)))
model.add(Conv3D(256,kernel_size=(3,3,3),activation='relu',padding="same",kernel_initializer=initializers.RandomNormal(stddev=0.01)))
model.add(BatchNormalization())
model.add(MaxPooling3D(pool_size=(2,2,2)))
#model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(512,activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(5))
model.summary()

In [ ]:
import os
checkpoint_path="/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/decay/0.001/training_model_weights/cp-{epoch:03d}.ckpt"
#os.makedirs("/content/drive/MyDrive/image/porosity_filtering/training_5layers", exist_ok=True)
#ckpt_callback = ModelCheckpoint(filepath='/content/drive/MyDrive/image/porosity_filtering/training_model_5layers/weights.{epoch:02d}-{val_loss:.2f}.hdf5', monitor='val_loss')
cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
import os
checkpoint_path="/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-{epoch:03d}.ckpt"
cp_callback=ModelCheckpoint(filepath=checkpoint_path,save_weights_only=True,verbose=1)

In [ ]:
csvlogger=CSVLogger('/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_185.log')

In [ ]:
class switchoptimizer(Callback):
  def __init__(self,switch_epoch):
    super(switchoptimizer,self).__init__()
    self.switch_epoch = switch_epoch
  def on_epoch_begin(self,epoch,logs=None):
    if epoch > self.switch_epoch:
      new_optimizer= SGD(learning_rate=0.01)
      self.model.compile(optimizer=new_optimizer, loss='mse',metrics=['mse'])


In [ ]:
class switchoptimizer(Callback):
  def __init__(self,switch_epoch):
    super(switchoptimizer,self).__init__()
    self.switch_epoch = switch_epoch
  def on_epoch_begin(self,epoch,logs=None):
    if epoch > self.switch_epoch:
      new_optimizer= SGD(learning_rate=0.01, momentum=0.8, decay=0.1,nesterov=False)
      self.model.compile(optimizer=new_optimizer, loss='mse',metrics=['mse'])


In [ ]:
class switchoptimizer(Callback):
  def __init__(self,switch_epoch):
    super(switchoptimizer,self).__init__()
    self.switch_epoch = switch_epoch
  def on_epoch_begin(self,epoch,epoch_prim,logs=None):
    if epoch > self.switch_epoch:
      def step_decay(epoch_prim):
        initial_lrate = 0.01
        drop = 0.5
        epochs_drop = 10.0
        lrate = initial_lrate * math.pow(drop,math.floor((epoch_prim)/epochs_drop))
        return lrate
      lrate=step_decay(epoch_prim)
      new_optimizer= SGD(learning_rate=lrate, momentum=0.8,nesterov=False)
      self.model.compile(optimizer=new_optimizer, loss='mse',metrics=['mse'])
      epoch_prim=epoch_prim+1

In [ ]:
model

In [ ]:
#این کلاس هست

In [ ]:
class switchoptimizer(Callback):
    def __init__(self, switch_epoch, model):
        super(switchoptimizer, self).__init__()
        self.switch_epoch = switch_epoch
        self.model= model

    def on_epoch_begin(self, epoch, logs=None):
        if epoch > self.switch_epoch:
            def step_decay(epoch):
                initial_lrate = 0.01
                drop = 0.5
                epochs_drop = 10.0
                #epoch_prim = epoch - 50
                lrate = initial_lrate * math.pow(drop, math.floor((epoch-50) / epochs_drop))
                return lrate

            lrate = step_decay(epoch)
            new_optimizer = SGD(learning_rate=lrate, momentum=0.8, nesterov=False)
            checkpoint_filename = checkpoint_path.format(epoch=epoch-1)
            self.model.compile(optimizer=new_optimizer, loss='mse', metrics=['mse'])
            self.model.load_weights(checkpoint_filename)


In [ ]:
class Changer(tf.keras.callbacks.Callback):
  def __init__(self, switch_epoch, model):
        super(Changer, self).__init__()
        self.switch_epoch = switch_epoch
        self.model= model
  def on_epoch_end(self, epoch, logs):
    if epoch > 50:
       def step_decay(epoch):
                initial_lrate = 0.001
                drop = 0.5
                epochs_drop = 10.0
                #epoch_prim = epoch - 50
                lrate = initial_lrate * math.pow(drop, math.floor((epoch-50) / epochs_drop))
                return lrate

       lrate = step_decay(epoch)
       self.model.optimizer = SGD(learning_rate=lrate, momentum=0.8, nesterov=False)

In [ ]:
class switchoptimizer(Callback):
  def __init__(self,switch_epoch):
    super(switchoptimizer,self).__init__()
    self.switch_epoch = switch_epoch
  def on_epoch_begin(self,epoch,logs=None):
    if epoch > self.switch_epoch:
      new_optimizer= SGD(learning_rate=0.01, momentum=0.8, decay=0.1,nesterov=False)
      self.model.compile(optimizer=new_optimizer, loss='mse',metrics=['mse'])
############################################
def step_decay(epoch,switch_epoch,epoch_prim):
  if epoch > switch_epoch:
        initial_lrate = 0.01
        drop = 0.5
        epochs_drop = 10.0
        lrate = initial_lrate * math.pow(drop,math.floor((1+epoch_prim)/epochs_drop))
        epoch_prim=epoch_prim+1
        return lrate
lr_scheduler=LearningRateScheduler(step_decay)

In [ ]:
class switchoptimizer(Callback):
  def __init__(self,switch_epoch,model):
    super(switchoptimizer,self).__init__()
    self.switch_epoch = switch_epoch
    self.model=model
  def on_epoch_begin(self,epoch,logs):
      if epoch > self.switch_epoch:
         def exp_decay(epoch):
             initial_lrate = 0.001
             k = 0.1
             lrate = initial_lrate * math.exp(-k*(epoch-50))
             return lrate
         lrate=exp_decay(epoch)
         self.model.optimizer = SGD(learning_rate=lrate, momentum=0.8, nesterov=False)



In [ ]:
switch_epoch=50
drop=0.5

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(opt,loss='mse',metrics=['mse'])

In [ ]:
lr_scheduler = switchoptimizer(switch_epoch,model)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler])

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-008.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=8)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-035.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger],initial_epoch=35)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-050.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=50)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-050.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=2, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=50)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-050.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=51)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-089.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=89)

In [ ]:
#floor

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-059.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=59)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-074.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=74)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-094.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=94)

In [ ]:
#0.001

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-050.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=50)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-093.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=93)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-104.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=104)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-123.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=123)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-143.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=143)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-152.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=152)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-155.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=155)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-167.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=167)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/floor/0.001/training_model_weights/cp-177.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=177)

In [ ]:
#exppppppppppppppppppppppppppppppppppppppppppppp

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/Decay/training_model_weights/cp-050.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=50)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-090.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=90)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-096.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=96)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-105.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=105)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-111.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=111)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-154.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=154)

In [ ]:
wieght='/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/exp/training_model_weights/cp-185.ckpt'
model.load_weights(wieght)

In [ ]:
history=model.fit_generator(train_gen, steps_per_epoch=nbatches_train, epochs=200, verbose=1, validation_data=valid_gen, validation_steps=nbatches_valid,callbacks=[cp_callback,csvlogger,lr_scheduler],initial_epoch=185)

In [ ]:
#############plotting_curve##########

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/learning rates.xlsx')

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
data

In [ ]:
fig= plt.figure(figsize=(10,7),dpi=200)
plt.semilogy(data.epoch,data.val_loss_decay,label= 'Sce1')
plt.semilogy(data.epoch,data.val_loss_floor,label = 'Sce2')
plt.semilogy(data.epoch,data.val_loss_exp,label= 'Sce3')
plt.semilogy(data.epoch,data['val_loss_0.0005'],label = '0.0005')
plt.semilogy(data.epoch,data['val_mse_0.003'],label = '0.003')
plt.semilogy(data.epoch,data['val_mse_0.001'],label = '0.001')
plt.xlabel('epoch', fontsize = 20)
plt.ylabel('loss',  fontsize = 20)
plt.xticks (fontsize = 15)
plt.yticks (fontsize = 15)
plt.legend (fontsize = 20)
plt.savefig('/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/learning_rate.png')

In [ ]:
data = pd.read_excel('/content/drive/MyDrive/image/porosity_filtering/learning_rate/learning rates_revised.xlsx')

In [ ]:
data

In [ ]:
fig= plt.figure(figsize=(10,7),dpi=200)
plt.semilogy(data.epoch,data.val_loss_decay,label= 'Sce1')
plt.semilogy(data.epoch,data.val_loss_floor,label = 'Sce2')
plt.semilogy(data.epoch,data.val_loss_exp,label= 'Sce3')
plt.semilogy(data.epoch,data['val_loss_0.0005'],label = '0.0005')
plt.semilogy(data.epoch,data['val_mse_0.003'],label = '0.003')
plt.semilogy(data.epoch,data['val_mse_0.001'],label = '0.001')
plt.xlabel('epoch', fontsize = 20)
plt.ylabel('loss',  fontsize = 20)
plt.xticks (fontsize = 15)
plt.yticks (fontsize = 15)
plt.legend (fontsize = 20)
plt.savefig('/content/drive/MyDrive/image/porosity_filtering/learning_rate/Adam_to_SGD/learning_rate_revised.png')